In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
import numpy as np
from scipy import stats
from scipy.spatial.distance import mahalanobis

class AnomalyDetectionSuite:
    """
    Comprehensive suite of anomaly detection algorithms for hyperspectral data
    """
    
    def __init__(self):
        self.algorithms = {}
        self.results = {}
        self.hyperparameters = {}
    
    def kmeans_anomaly_detection(self, X_train, X_test, n_clusters_range=(2, 10), random_state=42):
        """
        K-Means based anomaly detection with hyperparameter tuning
        
        Justification:
        - Assumes normal pixels form dense clusters
        - Anomalies are far from cluster centers
        - Effective for detecting spectral outliers
        """
        print("\n🔍 K-Means Anomaly Detection")
        print("Justification: Normal pixels form clusters; anomalies are distant from centers")
        
        # Hyperparameter tuning for optimal number of clusters
        best_score = -1
        best_k = 2
        silhouette_scores = []
        
        print("Tuning number of clusters...")
        for k in range(n_clusters_range[0], n_clusters_range[1] + 1):
            kmeans = KMeans(n_clusters=k, random_state=random_state, n_init=10)
            labels = kmeans.fit_predict(X_train)
            
            if len(np.unique(labels)) > 1:  # Need at least 2 clusters for silhouette
                score = silhouette_score(X_train, labels)
                silhouette_scores.append((k, score))
                
                if score > best_score:
                    best_score = score
                    best_k = k
        
        print(f"Optimal clusters: {best_k} (silhouette score: {best_score:.4f})")
        
        # Train final model with optimal parameters
        kmeans = KMeans(n_clusters=best_k, random_state=random_state, n_init=10)
        kmeans.fit(X_train)
        
        # Calculate anomaly scores (distance to nearest cluster center)
        train_distances = np.min(kmeans.transform(X_train), axis=1)
        test_distances = np.min(kmeans.transform(X_test), axis=1)
        
        # Normalize scores to [0, 1]
        all_distances = np.concatenate([train_distances, test_distances])
        min_dist, max_dist = np.min(all_distances), np.max(all_distances)
        
        train_scores = (train_distances - min_dist) / (max_dist - min_dist + 1e-8)
        test_scores = (test_distances - min_dist) / (max_dist - min_dist + 1e-8)
        
        self.algorithms['kmeans'] = kmeans
        self.hyperparameters['kmeans'] = {
            'n_clusters': best_k,
            'silhouette_score': best_score,
            'silhouette_scores': silhouette_scores
        }
        
        return train_scores, test_scores
    
    def rx_anomaly_detection(self, X_train, X_test):
        """
        Reed-Xiaoli (RX) anomaly detector
        
        Justification:
        - Statistical detector based on Mahalanobis distance
        - Assumes multivariate Gaussian background
        - Widely used standard in hyperspectral anomaly detection
        """
        print("\n🔍 RX (Reed-Xiaoli) Anomaly Detection")
        print("Justification: Statistical detector using Mahalanobis distance from background")
        
        # Calculate background statistics from training data
        mean_background = np.mean(X_train, axis=0)
        cov_background = np.cov(X_train, rowvar=False)
        
        # Add regularization to avoid singular covariance matrix
        reg_param = 1e-6
        cov_background += reg_param * np.eye(cov_background.shape[0])
        
        try:
            cov_inv = np.linalg.inv(cov_background)
        except np.linalg.LinAlgError:
            print("Using pseudo-inverse due to singular covariance matrix")
            cov_inv = np.linalg.pinv(cov_background)
        
        # Calculate RX scores (Mahalanobis distances)
        def calculate_rx_scores(X):
            diff = X - mean_background
            rx_scores = np.sum(diff @ cov_inv * diff, axis=1)
            return rx_scores
        
        train_rx_scores = calculate_rx_scores(X_train)
        test_rx_scores = calculate_rx_scores(X_test)
        
        # Normalize scores to [0, 1]
        all_scores = np.concatenate([train_rx_scores, test_rx_scores])
        min_score, max_score = np.min(all_scores), np.max(all_scores)
        
        train_scores = (train_rx_scores - min_score) / (max_score - min_score + 1e-8)
        test_scores = (test_rx_scores - min_score) / (max_score - min_score + 1e-8)
        
        self.algorithms['rx'] = {
            'mean_background': mean_background,
            'cov_inv': cov_inv
        }
        
        return train_scores, test_scores
    
    def pca_anomaly_detection(self, X_train, X_test, n_components_range=(0.95, 0.99)):
        """
        PCA-based anomaly detection with hyperparameter tuning
        
        Justification:
        - Normal data lies in lower-dimensional subspace
        - Anomalies have large reconstruction errors
        - Effective for detecting spectral anomalies
        """
        print("\n🔍 PCA-based Anomaly Detection")
        print("Justification: Anomalies have large reconstruction errors from principal subspace")
        
        # Hyperparameter tuning for number of components
        best_score = float('inf')
        best_components = 0.95
        component_scores = []
        
        print("Tuning number of PCA components...")
        for var_ratio in np.arange(n_components_range[0], n_components_range[1] + 0.01, 0.01):
            pca = PCA(n_components=var_ratio, random_state=42)
            X_train_pca = pca.fit_transform(X_train)
            X_train_reconstructed = pca.inverse_transform(X_train_pca)
            
            # Calculate reconstruction error
            reconstruction_error = np.mean(np.sum((X_train - X_train_reconstructed)**2, axis=1))
            component_scores.append((var_ratio, pca.n_components_, reconstruction_error))
            
            if reconstruction_error < best_score:
                best_score = reconstruction_error
                best_components = var_ratio
        
        print(f"Optimal variance ratio: {best_components:.2f} (reconstruction error: {best_score:.6f})")
        
        # Train final PCA model
        pca = PCA(n_components=best_components, random_state=42)
        X_train_pca = pca.fit_transform(X_train)
        X_test_pca = pca.transform(X_test)
        
        # Reconstruct data
        X_train_reconstructed = pca.inverse_transform(X_train_pca)
        X_test_reconstructed = pca.inverse_transform(X_test_pca)
        
        # Calculate reconstruction errors as anomaly scores
        train_errors = np.sum((X_train - X_train_reconstructed)**2, axis=1)
        test_errors = np.sum((X_test - X_test_reconstructed)**2, axis=1)
        
        # Normalize scores to [0, 1]
        all_errors = np.concatenate([train_errors, test_errors])
        min_error, max_error = np.min(all_errors), np.max(all_errors)
        
        train_scores = (train_errors - min_error) / (max_error - min_error + 1e-8)
        test_scores = (test_errors - min_error) / (max_error - min_error + 1e-8)
        
        self.algorithms['pca'] = pca
        self.hyperparameters['pca'] = {
            'n_components': pca.n_components_,
            'explained_variance_ratio': best_components,
            'reconstruction_error': best_score,
            'component_scores': component_scores
        }
        
        return train_scores, test_scores
    
    def isolation_forest_detection(self, X_train, X_test, contamination_range=(0.01, 0.1)):
        """
        Isolation Forest anomaly detection with hyperparameter tuning
        
        Justification:
        - Ensemble method that isolates anomalies
        - No assumptions about data distribution
        - Effective for high-dimensional data
        """
        print("\n🔍 Isolation Forest Anomaly Detection")
        print("Justification: Ensemble method that isolates anomalies in random subspaces")
        
        # Hyperparameter tuning for contamination parameter
        best_score = -1
        best_contamination = 0.05
        contamination_scores = []
        
        print("Tuning contamination parameter...")
        for contamination in np.arange(contamination_range[0], contamination_range[1] + 0.01, 0.01):
            iso_forest = IsolationForest(
                contamination=contamination,
                random_state=42,
                n_estimators=100
            )
            iso_forest.fit(X_train)
            
            # Calculate anomaly scores
            train_scores_raw = iso_forest.decision_function(X_train)
            
            # Use negative mean score as optimization metric (higher is better)
            score = -np.mean(train_scores_raw)
            contamination_scores.append((contamination, score))
            
            if score > best_score:
                best_score = score
                best_contamination = contamination
        
        print(f"Optimal contamination: {best_contamination:.3f} (score: {best_score:.4f})")
        
        # Train final model
        iso_forest = IsolationForest(
            contamination=best_contamination,
            random_state=42,
            n_estimators=100
        )
        iso_forest.fit(X_train)
        
        # Get anomaly scores (negative values indicate anomalies)
        train_scores_raw = iso_forest.decision_function(X_train)
        test_scores_raw = iso_forest.decision_function(X_test)
        
        # Convert to anomaly scores (higher = more anomalous)
        train_scores = -train_scores_raw
        test_scores = -test_scores_raw
        
        # Normalize to [0, 1]
        all_scores = np.concatenate([train_scores, test_scores])
        min_score, max_score = np.min(all_scores), np.max(all_scores)
        
        train_scores = (train_scores - min_score) / (max_score - min_score + 1e-8)
        test_scores = (test_scores - min_score) / (max_score - min_score + 1e-8)
        
        self.algorithms['isolation_forest'] = iso_forest
        self.hyperparameters['isolation_forest'] = {
            'contamination': best_contamination,
            'n_estimators': 100,
            'contamination_scores': contamination_scores
        }
        
        return train_scores, test_scores

# Initialize anomaly detection suite
anomaly_suite = AnomalyDetectionSuite()
print("✅ Anomaly Detection Suite initialized")